# nb158 — Boltz2 smoke test (5 compounds, max diagnostic)

Validates install + single-compound prediction. Should finish in 30-60 min.
If this works, nb135 v9 is on the right track. If not, fix here and re-push 135.

In [ ]:
import os, subprocess, sys, urllib.request, time, json
from pathlib import Path
os.environ['PYTHONUNBUFFERED'] = '1'

BZ_TARGET = '/kaggle/working/boltz_pkgs'
Path(BZ_TARGET).mkdir(exist_ok=True)
BOLTZ_BIN = f'{BZ_TARGET}/bin/boltz'

if not Path(BOLTZ_BIN).exists():
    print('Installing boltz to isolated --target...')
    r = subprocess.run([sys.executable, '-m', 'pip', 'install',
                    '--target', BZ_TARGET, '-q',
                    'numpy==1.26.4', 'pandas==2.2.3', 'scipy==1.13.1',
                    'torch==2.4.0', 'torchmetrics==1.4.0', 'lightning==2.4.0',
                    'boltz', 'rdkit', 'pyyaml'], capture_output=True, text=True)
    print('Install RC:', r.returncode)
    print('Install STDERR tail:', r.stderr[-1500:])
    bin_dir = Path(f'{BZ_TARGET}/bin')
    if bin_dir.exists():
        for f in bin_dir.iterdir(): f.chmod(0o755)
print('boltz bin exists:', Path(BOLTZ_BIN).exists())

In [ ]:
env = {**os.environ, 'PYTHONPATH': BZ_TARGET, 'PATH': f'{BZ_TARGET}/bin:' + os.environ.get('PATH','')}
r = subprocess.run([sys.executable, '-c',
    'import sys; sys.path.insert(0, "/kaggle/working/boltz_pkgs"); import numpy, pandas, boltz; print(f"numpy={numpy.__version__} boltz_OK")'],
    capture_output=True, text=True, timeout=120)
print('STDOUT:', r.stdout)
print('STDERR tail:', r.stderr[-1000:] if r.stderr else '')

In [ ]:
# Write 5 test YAMLs
import yaml
PXR_SEQ = ('LDRRTVVPATQHVTGTAYIWYRSGLCEHHIVEAATRGNVMTPSCKLITEELLGRPVHIVQPVKAVCS'
           'IVKQSDCRPFNQRSFKKYFTMENKVMVLNQELIKLALNFKLQDGRPHGGIIYDLSGEEDPKSWIWE'
           'VLEAWDIKAQVGPVTYAVTSLPFLQLSQYLDQDLALYIHQAFRYGPNALLDLLTDTRKHADRLELN'
           'GLAIRLLPELEVALMLLTQHTLREEKAGNFETIAEPFNALVMQVMEGYREKDPEAKQNQELHIWAN'
           'KTKDPLLLEAHALDQFSCK')
TEST_SMILES = [
    ('rifampicin', 'CC1C(C(C(C=C(C(C(C=CC=C(C(C(C2=C(C3=C(C(=C2O)C)O)C(=O)C=C(N3)C)/C)O)C)OC(=O)C)C)O)C)O)C(=O)O1'),
    ('hyperforin', 'CC(=CCCC(C)(C(C(=O)C(C=C)(C)C)O)CC=C(C)C)C'),
    ('paclitaxel', 'CC1=C2C(C(=O)C3(C(CC4C(C3C(C(C2(C)C)(CC1OC(=O)C(C(C5=CC=CC=C5)NC(=O)C6=CC=CC=C6)O)O)OC(=O)C7=CC=CC=C7)(CO4)OC(=O)C)O)C)O'),
    ('mifepristone', 'CC#CC1(CCC2C1(CC(C3=C4CCC(=O)CC4=CCC23)C5=CC=C(C=C5)N(C)C)C)O'),
    ('sr12813', 'CCOP(=O)(CC(c1cc(c(c(c1)C(C)(C)C)O)C(C)(C)C)P(=O)(OCC)OCC)OCC'),
]
YAML_DIR = Path('/kaggle/working/y'); YAML_DIR.mkdir(exist_ok=True)
for name, smi in TEST_SMILES:
    cfg = {'version': 1, 'sequences': [{'protein': {'id': 'A', 'sequence': PXR_SEQ}}, {'ligand': {'id': 'B', 'smiles': smi}}], 'properties': [{'affinity': {'binder': 'B'}}]}
    (YAML_DIR / f'{name}.yaml').write_text(yaml.safe_dump(cfg, sort_keys=False))
print('Wrote 5 YAMLs')

In [ ]:
# Run boltz on first compound with full stderr
import time
OUT = Path('/kaggle/working/o'); OUT.mkdir(exist_ok=True)
def find_aff(d):
    for jf in Path(d).glob('**/*affinity*.json'):
        try:
            j = json.load(open(jf))
            return j.get('affinity_pred_value') or j.get('affinity_pred') or j.get('affinity')
        except Exception:
            pass
    return None

results = []
for name, _ in TEST_SMILES:
    y_in = YAML_DIR / f'{name}.yaml'
    out_p = OUT / name
    cmd = [BOLTZ_BIN, 'predict', str(y_in), '--out_dir', str(out_p), '--use_msa_server',
           '--diffusion_samples', '1', '--recycling_steps', '1', '--sampling_steps', '50']
    print(f'\n=== {name} ===')
    print('CMD:', ' '.join(cmd))
    t0 = time.time()
    r = subprocess.run(cmd, env=env, capture_output=True, text=True, timeout=2400)
    dt = time.time() - t0
    print(f'  rc={r.returncode}  elapsed={dt/60:.1f}min')
    if r.returncode != 0:
        print('  STDERR tail:', r.stderr[-1500:])
    aff = find_aff(out_p)
    print(f'  affinity: {aff}')
    results.append({'name': name, 'aff': aff, 'rc': r.returncode, 'elapsed_min': dt/60})

import pandas as pd
df = pd.DataFrame(results)
df.to_parquet('/kaggle/working/nb158_smoketest.parquet', index=False)
print('\nFinal:')
print(df)